In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import seaborn as sns


# ---------------------------------------------------------
# Helper: Fit GARCH(1,1) via Scipy SLSQP (Fallback Solver)
# ---------------------------------------------------------
def fit_garch_scipy(returns):
  scale = 100.0
  r = returns.values * scale
  mu = np.mean(r)
  eps = r - mu
  var_eps = np.var(eps)

  def neg_log_lik(params):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or (alpha + beta) >= 1.0:
      return 1e10
    n = len(eps)
    sigma2 = np.zeros(n)
    sigma2[0] = var_eps
    for t in range(1, n):
      sigma2[t] = omega + alpha * (eps[t - 1] ** 2) + beta * sigma2[t - 1]
      if sigma2[t] <= 0:
        sigma2[t] = 1e-6
    return 0.5 * np.sum(np.log(2 * np.pi * sigma2) + (eps**2) / sigma2)

  bounds = [(1e-6, 1.0), (1e-6, 0.49), (1e-6, 0.99)]
  constraints = {'type': 'ineq', 'fun': lambda p: 0.999 - (p[1] + p[2])}

  init_p = [var_eps * 0.05, 0.08, 0.88]
  res = minimize(
      neg_log_lik,
      init_p,
      method='SLSQP',
      bounds=bounds,
      constraints=constraints,
  )

  omega_s, alpha, beta = res.x
  n = len(eps)
  sigma2_s = np.zeros(n)
  sigma2_s[0] = var_eps
  for t in range(1, n):
    sigma2_s[t] = omega_s + alpha * (eps[t - 1] ** 2) + beta * sigma2_s[t - 1]

  omega = omega_s / (scale**2)
  cond_vol = pd.Series(np.sqrt(sigma2_s) / scale, index=returns.index)

  eps_orig = returns.values - (mu / scale)
  std_residuals = pd.Series(eps_orig / cond_vol.values, index=returns.index)

  forecast_var = omega + alpha * (eps_orig**2) + beta * (cond_vol.values**2)
  forecast_vol = pd.Series(np.sqrt(forecast_var), index=returns.index)

  return {
      'mu': mu / scale,
      'omega': omega,
      'alpha': alpha,
      'beta': beta,
      'cond_vol': cond_vol,
      'std_residuals': std_residuals,
      'forecast_vol': forecast_vol,
  }


def run_phase_6():
  print("==================================================")
  print("      RUNNING PHASE 6: GARCH VOLATILITY SCALING   ")
  print("==================================================")

  # ---------------------------------------------------------
  # 1. Load Data
  # ---------------------------------------------------------
  input_file = 'portfolio_phase3_output(10).csv'
  if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"'{input_file}' not found! Please run Phase 3 first."
    )

  df_p3 = pd.read_csv(input_file, parse_dates=['Date'], index_col='Date')
  returns = df_p3['Daily_Return']
  pnl = df_p3['Daily_PnL']
  dates = df_p3.index

  window = 250
  conf_level = 0.99
  alpha_var = 1.0 - conf_level

  # ---------------------------------------------------------
  # 2. Fit GARCH(1,1) Model
  # ---------------------------------------------------------
  print(' Fitting GARCH(1,1) model on portfolio returns...')

  try:
    from arch import arch_model

    am = arch_model(returns * 100.0, vol='Garch', p=1, q=1, dist='normal')
    res = am.fit(disp='off')

    mu = res.params['mu'] / 100.0
    omega = res.params['omega'] / (100.0**2)
    alpha_g = res.params['alpha[1]']
    beta_g = res.params['beta[1]']

    cond_vol = res.conditional_volatility / 100.0
    std_residuals = res.std_resid

    # Forecast 1-day ahead
    forecasts = res.forecast(horizon=1)
    forecast_var = forecasts.variance.values[-1, 0] / (100.0**2)
    # Reconstruct 1-day ahead time-series forecast
    eps = (returns - mu).values
    forecast_var_series = omega + alpha_g * (eps**2) + beta_g * (cond_vol**2)
    forecast_vol = pd.Series(np.sqrt(forecast_var_series), index=dates)

    print(' [✓] Fitted via arch package.')

  except ImportError:
    print(
        ' [!] arch package not found. Using internal Scipy MLE optimization...'
    )
    fit_res = fit_garch_scipy(returns)
    mu, omega, alpha_g, beta_g = (
        fit_res['mu'],
        fit_res['omega'],
        fit_res['alpha'],
        fit_res['beta'],
    )
    cond_vol = fit_res['cond_vol']
    std_residuals = fit_res['std_residuals']
    forecast_vol = fit_res['forecast_vol']
    print(' [✓] Fitted via Scipy MLE.')

  print(
      f'     Params -> Omega: {omega:.8f} | Alpha: {alpha_g:.4f} | Beta:'
      f' {beta_g:.4f}'
  )
  print(f'     Persistence (Alpha + Beta): {alpha_g + beta_g:.4f}')

  # ---------------------------------------------------------
  # 3. Scale Returns & Compute GARCH-Scaled VaR
  # ---------------------------------------------------------
  raw_var_series = pd.Series(index=dates, dtype=float)
  garch_var_series = pd.Series(index=dates, dtype=float)

  # Latest day scaled returns (for chart 2 visualization)
  scaled_returns_latest = returns * (forecast_vol.iloc[-1] / cond_vol)

  print(' Calculating rolling GARCH-scaled Historical VaR...')
  for i in range(window, len(dates)):
    curr_forecast = forecast_vol.iloc[i]

    win_pnl = pnl.iloc[i - window : i]
    win_vols = cond_vol.iloc[i - window : i]

    # Raw Historical VaR
    raw_var = -np.percentile(win_pnl, alpha_var * 100)
    raw_var_series.iloc[i] = raw_var

    # GARCH-Scaled Historical VaR
    scaled_pnl = win_pnl * (curr_forecast / win_vols)
    garch_var = -np.percentile(scaled_pnl, alpha_var * 100)
    garch_var_series.iloc[i] = garch_var

  # ---------------------------------------------------------
  # 4. Save Outputs to CSV
  # ---------------------------------------------------------
  output_df = pd.DataFrame(
      {
          'Conditional_Volatility': cond_vol,
          'Forecast_Volatility': forecast_vol,
          'Standardized_Residuals': std_residuals,
          'Raw_99_VaR': raw_var_series,
          'GARCH_Scaled_99_VaR': garch_var_series,
      },
      index=dates,
  )

  output_df.to_csv('garch_scaled_var_summary_phase6(10).csv')
  print(' [✓] Saved Data -> garch_scaled_var_summary_phase6(10).csv')

  # ---------------------------------------------------------
  # 5. Generate & Save Charts
  # ---------------------------------------------------------
  sns.set_theme(style='whitegrid')

  # CHART 1: GARCH Conditional Volatility
  plt.figure(figsize=(12, 5))
  plt.plot(
      cond_vol.index,
      cond_vol * np.sqrt(252),
      color='navy',
      lw=1.5,
      label='Annualized GARCH(1,1) Conditional Volatility (σ_t)',
  )
  plt.title('Chart 1: GARCH(1,1) Annualized Conditional Volatility')
  plt.xlabel('Date')
  plt.ylabel('Annualized Volatility (%)')
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart1_garch_conditional_volatility(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart1_garch_conditional_volatility(10).png')

  # CHART 2: Forecast Volatility vs Conditional Volatility
  plt.figure(figsize=(12, 5))
  plt.plot(
      cond_vol.index,
      cond_vol * np.sqrt(252),
      color='navy',
      alpha=0.7,
      lw=1.2,
      label='Conditional Volatility (σ_t)',
  )
  plt.plot(
      forecast_vol.index,
      forecast_vol * np.sqrt(252),
      color='crimson',
      lw=1.2,
      linestyle='--',
      label='1-Day Ahead Forecast Volatility (σ_{t+1})',
  )
  plt.title(
      'Chart 2: GARCH(1,1) Conditional Volatility vs. 1-Day Forecast'
      ' Volatility'
  )
  plt.xlabel('Date')
  plt.ylabel('Annualized Volatility (%)')
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart2_garch_forecast_volatility(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart2_garch_forecast_volatility(10).png')

  # CHART 3: Historical VaR vs GARCH-Scaled VaR
  plt.figure(figsize=(12, 6))
  plt.plot(
      raw_var_series.index,
      raw_var_series,
      label='Raw 99% Historical VaR ($)',
      color='darkorange',
      lw=1.5,
      linestyle='--',
  )
  plt.plot(
      garch_var_series.index,
      garch_var_series,
      label='GARCH-Scaled 99% Historical VaR ($)',
      color='forestgreen',
      lw=1.8,
  )
  plt.title('Chart 3: Rolling 250-Day Historical VaR vs. GARCH(1,1)-Scaled VaR')
  plt.xlabel('Date')
  plt.ylabel('1-Day Potential Loss ($)')
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart3_historical_vs_garch_var(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart3_historical_vs_garch_var(10).png')

  # CHART 4: GARCH Standardized Residuals
  fig, (ax1, ax2) = plt.subplots(
      2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [2, 1]}
  )
  ax1.plot(
      std_residuals.index, std_residuals, color='teal', lw=0.8, alpha=0.8
  )
  ax1.axhline(0, color='black', linestyle='--', lw=1)
  ax1.axhline(2, color='red', linestyle=':', lw=1, label='±2 Std Dev')
  ax1.axhline(-2, color='red', linestyle=':', lw=1)
  ax1.set_title(
      'Chart 4: GARCH(1,1) Standardized Residuals (z_t = ε_t / σ_t)'
  )
  ax1.set_ylabel('Standardized Residual')
  ax1.legend()

  sns.histplot(std_residuals, kde=True, ax=ax2, color='teal', bins=60)
  ax2.set_title('Distribution of Standardized Residuals')
  ax2.set_xlabel('Residual Value')
  plt.tight_layout()
  plt.savefig('chart4_garch_standardized_residuals(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart4_garch_standardized_residuals(10).png')

  # CHART 5: Volatility Clustering Chart
  plt.figure(figsize=(12, 5))
  plt.plot(
      returns.index,
      returns**2,
      color='gray',
      alpha=0.5,
      lw=0.8,
      label='Squared Portfolio Returns (r_t²)',
  )
  plt.plot(
      cond_vol.index,
      cond_vol**2,
      color='red',
      lw=1.8,
      label='GARCH(1,1) Conditional Variance (σ_t²)',
  )
  plt.title(
      'Chart 5: Volatility Clustering (Squared Returns vs. GARCH Conditional'
      ' Variance)'
  )
  plt.xlabel('Date')
  plt.ylabel('Daily Variance')
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart5_volatility_clustering(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart5_volatility_clustering(10).png')

  print('\n==================================================')
  print('          PHASE 6 COMPLETED SUCCESSFULLY!         ')
  print('==================================================')


if __name__ == '__main__':
  run_phase_6()

      RUNNING PHASE 6: GARCH VOLATILITY SCALING   
 Fitting GARCH(1,1) model on portfolio returns...
 [!] arch package not found. Using internal Scipy MLE optimization...
 [✓] Fitted via Scipy MLE.
     Params -> Omega: 0.00000484 | Alpha: 0.1470 | Beta: 0.8126
     Persistence (Alpha + Beta): 0.9596
 Calculating rolling GARCH-scaled Historical VaR...
 [✓] Saved Data -> garch_scaled_var_summary_phase6(10).csv
 [✓] Saved -> chart1_garch_conditional_volatility(10).png
 [✓] Saved -> chart2_garch_forecast_volatility(10).png
 [✓] Saved -> chart3_historical_vs_garch_var(10).png
 [✓] Saved -> chart4_garch_standardized_residuals(10).png
 [✓] Saved -> chart5_volatility_clustering(10).png

          PHASE 6 COMPLETED SUCCESSFULLY!         
